# ChronoState-KD Lite — Phase 0: Environment check

**Goal:** prove the free Colab T4 can run this project, install the stack, mount Drive, run the CPU unit tests, and freeze the exact package versions.

**Gate to pass before Phase 1:**
1. `nvidia-smi` shows a Tesla T4 (or better)
2. `torch.backends.cuda.mem_efficient_sdp_enabled()` is `True` (that is the attention kernel we use instead of FlashAttention-2, which needs Ampere)
3. `pytest tests/` is green
4. `requirements.lock.txt` and `results/env_report.json` are written and committed

**Before running:** Runtime → Change runtime type → **T4 GPU**. Then edit `GITHUB_URL` in the next cell.

In [ ]:
GITHUB_URL = "https://github.com/<YOUR_GITHUB_USER>/chronostate-kd.git"   # <-- edit: your public repo
REPO_DIR   = "/content/chronostate-kd"
DRIVE_ROOT = "/content/drive/MyDrive/chronostate"


## 1. GPU check

In [ ]:
!nvidia-smi
import torch, subprocess
assert torch.cuda.is_available(), "No GPU. Runtime -> Change runtime type -> T4 GPU, then restart."
name = torch.cuda.get_device_name(0)
cap  = torch.cuda.get_device_capability(0)
total_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"GPU: {name}  compute capability {cap}  VRAM {total_gb:.1f} GB")
print("bf16 supported:", torch.cuda.is_bf16_supported(), "(T4 -> False; we use float16 everywhere)")


## 2. Clone the repo and install
Colab already ships torch + CUDA; we install everything else on top of it and never reinstall torch.

In [ ]:
import os
if not os.path.exists(REPO_DIR):
    !git clone $GITHUB_URL $REPO_DIR
%cd $REPO_DIR
!git pull --ff-only
!pip install -q -r requirements.txt
!pip install -q -e .


## 3. Attention-kernel check
FlashAttention-2 requires sm_80+ (Ampere). On a T4 (sm_75) PyTorch's SDPA falls back to the **memory-efficient** kernel, which is what we want to see enabled.

In [ ]:
import torch, importlib
print("torch", torch.__version__, "| cuda", torch.version.cuda)
print("flash_sdp_enabled        :", torch.backends.cuda.flash_sdp_enabled(), "(will not be used on T4)")
print("mem_efficient_sdp_enabled:", torch.backends.cuda.mem_efficient_sdp_enabled(), "<- must be True")
print("math_sdp_enabled         :", torch.backends.cuda.math_sdp_enabled())
assert torch.backends.cuda.mem_efficient_sdp_enabled()

# quick SDPA smoke test in fp16 on the GPU
q = torch.randn(1, 8, 128, 64, device="cuda", dtype=torch.float16)
with torch.backends.cuda.sdp_kernel(enable_flash=False, enable_math=False, enable_mem_efficient=True):
    o = torch.nn.functional.scaled_dot_product_attention(q, q, q, is_causal=True)
print("SDPA mem-efficient fp16 OK:", o.shape, o.dtype)

for mod in ["transformers", "peft", "bitsandbytes", "accelerate", "datasets", "safetensors", "sentence_transformers"]:
    m = importlib.import_module(mod)
    print(f"{mod:22s} {getattr(m, '__version__', '?')}")


## 4. bitsandbytes 4-bit smoke test
Loads a tiny model in NF4 with `compute_dtype=float16` (bf16 is not available on Turing). If this works, the 3B teacher and 0.5B student will too.

In [ ]:
import torch, time
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
bnb = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4",
                         bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.float16)
tok = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct")
t0 = time.time()
m = AutoModelForCausalLM.from_pretrained("Qwen/Qwen2.5-0.5B-Instruct", quantization_config=bnb,
                                         attn_implementation="sdpa", device_map={"": 0})
print(f"loaded 0.5B NF4 in {time.time()-t0:.1f}s; VRAM {torch.cuda.memory_allocated()/1e9:.2f} GB")
msgs = [{"role": "user", "content": "Write a Python one-liner that reverses a string."}]
ids = tok.apply_chat_template(msgs, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = m.generate(ids, max_new_tokens=40, do_sample=False)
print(tok.decode(out[0][ids.shape[1]:], skip_special_tokens=True))
student_vocab = len(tok)
del m; torch.cuda.empty_cache()


## 5. Tokenizer compatibility (teacher ↔ student)
Logit distillation only makes sense if both models index the same vocabulary. This is the single most common silent failure, so it is asserted here and again in Phase 1.

In [ ]:
from transformers import AutoTokenizer
import yaml
cfg = yaml.safe_load(open("configs/lite.yaml"))
t_tok = AutoTokenizer.from_pretrained(cfg["models"]["teacher"])
s_tok = AutoTokenizer.from_pretrained(cfg["models"]["student"])
sample = "def fib(n):\n    return n if n < 2 else fib(n-1) + fib(n-2)  # 3.14159"
assert t_tok(sample)["input_ids"] == s_tok(sample)["input_ids"], "teacher/student tokenise differently!"
assert t_tok.get_vocab() == s_tok.get_vocab(), "vocab mismatch"
print("teacher/student tokenizers identical; vocab size", len(s_tok))


## 6. Mount Google Drive
Every phase checkpoints here so a dead session costs minutes, not hours.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
for sub in ["teacher_cache", "adapters", "ckpt", "results"]:
    os.makedirs(f"{DRIVE_ROOT}/{sub}", exist_ok=True)
!ls -la $DRIVE_ROOT


## 7. Unit tests (CPU)
The KD loss, Markov layer, checkpoint manager, monitor and config loader are all testable without a GPU.

In [ ]:
!python -m pytest tests/ -q


## 8. Freeze versions and write the environment report
Commit both files. Later notebooks install from `requirements.lock.txt` so Phase 2 runs on exactly the versions Phase 0 verified.

In [ ]:
import json, torch, subprocess, platform, importlib, time
!pip freeze > requirements.lock.txt
report = {
  "ts": time.time(),
  "gpu": torch.cuda.get_device_name(0),
  "compute_capability": list(torch.cuda.get_device_capability(0)),
  "vram_total_gb": round(torch.cuda.get_device_properties(0).total_memory/1e9, 2),
  "bf16": torch.cuda.is_bf16_supported(),
  "torch": torch.__version__, "cuda": torch.version.cuda, "python": platform.python_version(),
  "mem_efficient_sdp": torch.backends.cuda.mem_efficient_sdp_enabled(),
  "flash_sdp": torch.backends.cuda.flash_sdp_enabled(),
  "student_vocab": student_vocab,
  "packages": {m: importlib.import_module(m).__version__ for m in
               ["transformers","peft","bitsandbytes","accelerate","datasets","safetensors"]},
}
os.makedirs("results", exist_ok=True)
json.dump(report, open("results/env_report.json", "w"), indent=2)
json.dump(report, open(f"{DRIVE_ROOT}/results/env_report.json", "w"), indent=2)
print(json.dumps(report, indent=2))


## 9. Commit back to GitHub

Easiest: download `requirements.lock.txt` and `results/env_report.json` from the Files pane (left sidebar) and commit them from your laptop.

Or, from Colab with a GitHub token (Settings → Developer settings → Fine-grained token, repo contents: read/write), store it in Colab **Secrets** as `GH_TOKEN` and run:

In [ ]:
# from google.colab import userdata
# token = userdata.get("GH_TOKEN")
# !git config user.email "you@example.com" && git config user.name "Your Name"
# !git add requirements.lock.txt results/env_report.json && git commit -m "Phase 0: env report + lock file"
# !git push https://$token@github.com/<YOUR_GITHUB_USER>/chronostate-kd.git HEAD:main


## Gate checklist
- [ ] T4 shown in cell 1
- [ ] `mem_efficient_sdp_enabled` True in cell 3
- [ ] 0.5B NF4 model generated text in cell 4
- [ ] tokenizers identical in cell 5
- [ ] pytest green in cell 7
- [ ] `requirements.lock.txt` + `results/env_report.json` committed

Then update `PROGRESS.md` → Phase 0 → Outcome, and move on to `01_teacher_cache.ipynb`.